In [190]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare, ttest_rel
from scipy.stats import shapiro, friedmanchisquare, wilcoxon
import re
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go


In [142]:
df = pd.read_csv('VMIQ-2_AW_BCI.csv')
df = df.applymap(lambda x: str(x)[0] if isinstance(x, str) else x)
df

,ID,Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Caminar],Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Correr],Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Patear una piedra],Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Agacharse para recoger una moneda],Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Subir corriendo las escaleras],Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Saltar hacia un lado],Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Lanzar una piedra al agua],Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Patear una pelota en el aire],Verse a sí mismo realizando el movimiento (Imaginación visual externa) [Correr cuesta abajo],...,Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Patear una piedra],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Agacharse para recoger una moneda],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Subir corriendo las escaleras],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Saltar hacia un lado],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Lanzar una piedra al agua],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Patear una pelota en el aire],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Correr cuesta abajo],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Montar en bicicleta],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Balancearse en una cuerda],Sintiéndote a ti mismo haciendo el movimiento (Imaginación cinestésicas) [Saltar desde un muro alto]
0,1,5,5,5,5,5,5,5,5,5,...,5,5,5,5,5,5,5,5,5,5
1,2,3,3,3,3,3,3,3,3,3,...,4,4,4,4,4,4,4,4,4,4
2,3,4,4,5,5,5,4,5,5,5,...,5,5,5,4,5,5,5,4,3,5
3,4,5,5,5,5,5,5,5,5,5,...,4,5,5,5,5,4,5,5,5,5
4,5,4,5,5,3,4,3,5,3,3,...,5,5,5,3,5,5,5,5,3,5
5,6,5,5,4,5,5,3,5,4,5,...,5,5,5,4,5,3,5,5,4,5
6,7,4,3,4,4,3,4,2,2,4,...,4,5,5,3,3,3,4,3,2,2
7,8,5,5,4,5,5,5,4,3,4,...,4,4,4,2,2,3,4,4,3,4
8,9,3,5,5,4,4,4,4,3,3,...,5,5,5,5,5,5,5,2,4,5
9,10,5,5,4,3,2,3,2,1,2,...,3,3,2,2,4,2,2,3,5,5


In [143]:

# Extract text inside parentheses and create dictionary for translation
translation_dict = {
    'Imaginación visual externa': 'External_Visual',
    'Imaginación visual interna': 'Internal_Visual',
    'Imaginación cinestésicas': 'Kinesthetic'
}

# Create new column names by iterating over the full original column names
new_names = ['ID']  # Keep ID column as is
for full_col in df.columns[1:]:
    paren_match = re.search(r'\((.*)\)', full_col)

    spa = paren_match.group(1).strip() if paren_match else None

    if spa in translation_dict:
        eng = translation_dict[spa]
        new_col = eng
    else:
        # fallback: keep original column name (or sanitize if desired)
        new_col = full_col

    new_names.append(new_col)

# Ensure the number of new names matches the number of columns
if len(new_names) != len(df.columns):
    # As a safe fallback, keep original columns if mismatch occurs
    new_names = list(df.columns)

# Rename columns
df.columns = new_names
df

,ID,External_Visual,External_Visual,External_Visual,External_Visual,External_Visual,External_Visual,External_Visual,External_Visual,External_Visual,...,Kinesthetic,Kinesthetic,Kinesthetic,Kinesthetic,Kinesthetic,Kinesthetic,Kinesthetic,Kinesthetic,Kinesthetic,Kinesthetic
0,1,5,5,5,5,5,5,5,5,5,...,5,5,5,5,5,5,5,5,5,5
1,2,3,3,3,3,3,3,3,3,3,...,4,4,4,4,4,4,4,4,4,4
2,3,4,4,5,5,5,4,5,5,5,...,5,5,5,4,5,5,5,4,3,5
3,4,5,5,5,5,5,5,5,5,5,...,4,5,5,5,5,4,5,5,5,5
4,5,4,5,5,3,4,3,5,3,3,...,5,5,5,3,5,5,5,5,3,5
5,6,5,5,4,5,5,3,5,4,5,...,5,5,5,4,5,3,5,5,4,5
6,7,4,3,4,4,3,4,2,2,4,...,4,5,5,3,3,3,4,3,2,2
7,8,5,5,4,5,5,5,4,3,4,...,4,4,4,2,2,3,4,4,3,4
8,9,3,5,5,4,4,4,4,3,3,...,5,5,5,5,5,5,5,2,4,5
9,10,5,5,4,3,2,3,2,1,2,...,3,3,2,2,4,2,2,3,5,5


In [144]:
dict_VIQ={}
for col in df.columns[1:]:
    dict_VIQ[col]=df.loc[:, col].astype(float).mean(axis=1)

df_VIQ = pd.concat([df.ID, pd.DataFrame(dict_VIQ)], axis=1)
df_VIQ

,ID,External_Visual,Internal_Visual,Kinesthetic
0,1,4.833333,3.833333,5.000000
1,2,3.000000,4.000000,3.916667
2,3,4.666667,4.500000,4.500000
3,4,5.000000,5.000000,4.833333
4,5,3.833333,4.666667,4.666667
5,6,4.583333,5.000000,4.666667
6,7,3.083333,3.833333,3.666667
7,8,4.250000,4.416667,3.333333
8,9,3.166667,3.750000,4.500000
9,10,3.083333,3.500000,3.416667


In [145]:
df_NASA= pd.read_csv('NASA_TLX_BCI_AW.csv')

df_NASA 

,ID,TASK,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration
0,1,AW,140,500,160,0,300,70
1,2,AW,10,50,30,300,30,0
2,3,AW,200,0,240,240,120,40
3,4,AW,250,20,120,320,160,0
4,5,AW,400,180,50,180,320,0
...,...,...,...,...,...,...,...,...
82,25,MO,150,0,120,140,70,400
83,26,MO,350,160,160,60,200,70
84,27,MO,120,0,40,90,60,60
85,28,MO,240,0,60,320,160,10


In [128]:
vmiq_data = df_VIQ.drop(columns='ID')

print("Normality")

print("\n VMIQ-2:")
normality_results_vmiq = {}
for factor in vmiq_data.columns:
    data = vmiq_data.dropna()
    stat, p = shapiro(data)
    normality_results_vmiq[factor] = p
    print(f'  {factor}: W={stat:.4f}, p={p:.4f}{" (Normal)" if p > 0.05 else " (NO Normal)"}')

nasa_factors = df_NASA.drop(columns=['ID', 'TASK']).columns
tasks = df_NASA['TASK'].unique()

normality_results_nasa = {}
for factor in nasa_factors:
    print(f"\n  Factor: {factor}")
    is_normal = True
    for task in tasks:
        subset = df_NASA[df_NASA['TASK'] == task][factor].dropna()
        stat, p = shapiro(subset)
        if p <= 0.05:
            is_normal = False
        print(f'    {task} condition: W={stat:.4f}, p={p:.4f}{" (Normal)" if p > 0.05 else " (NO Normal)"}')
    normality_results_nasa[factor] = is_normal 



Normality

 VMIQ-2:
  External_Visual: W=0.9466, p=0.0013 (NO Normal)
  Internal_Visual: W=0.9466, p=0.0013 (NO Normal)
  Kinesthetic: W=0.9466, p=0.0013 (NO Normal)

  Factor: Mental Demand
    AW condition: W=0.9225, p=0.0352 (NO Normal)
    MI condition: W=0.9379, p=0.0882 (Normal)
    MO condition: W=0.9311, p=0.0587 (Normal)

  Factor: Physical Demand
    AW condition: W=0.6619, p=0.0000 (NO Normal)
    MI condition: W=0.8158, p=0.0002 (NO Normal)
    MO condition: W=0.7632, p=0.0000 (NO Normal)

  Factor: Temporal Demand
    AW condition: W=0.9383, p=0.0906 (Normal)
    MI condition: W=0.8949, p=0.0074 (NO Normal)
    MO condition: W=0.9163, p=0.0245 (NO Normal)

  Factor: Performance
    AW condition: W=0.9647, p=0.4272 (Normal)
    MI condition: W=0.9427, p=0.1179 (Normal)
    MO condition: W=0.9129, p=0.0203 (NO Normal)

  Factor: Effort
    AW condition: W=0.9347, p=0.0731 (Normal)
    MI condition: W=0.9489, p=0.1711 (Normal)
    MO condition: W=0.9229, p=0.0361 (NO Normal)


In [127]:
df_pivot = df.pivot(index='ID', columns='TASK', values=nasa_factors)

df_pivot.columns = ['_'.join(col).strip() for col in df_pivot.columns.values]

df_pivot = df_pivot.join(df_VIQ.set_index('ID')[vmiq_data.columns])

print("\n Significant test NASA")

for factor in nasa_factors:
    
    data_aw = df_pivot[f'{factor}_AW']
    data_mi = df_pivot[f'{factor}_MI']
    data_mo = df_pivot[f'{factor}_MO']
    

    print("\nFriedman")
    try:
        stat, p = friedmanchisquare(data_aw, data_mi, data_mo)
        print(f" Value={stat:.4f}, p-valor={p:.4f}")
        
        if p < 0.05:
            print("\Post-Hoc (Wilcoxon) ", factor)
            num_comparisons = 3
            
            stat_aw_mi, p_aw_mi = wilcoxon(data_aw, data_mi)
            p_adj_aw_mi = min(p_aw_mi * num_comparisons, 1.0) 
            print(f"AW vs MI: p={p_aw_mi:.4f}, p-ajustado={p_adj_aw_mi:.4f}{' *' if p_adj_aw_mi < 0.05 else ''}")
            
            stat_aw_mo, p_aw_mo = wilcoxon(data_aw, data_mo)
            p_adj_aw_mo = min(p_aw_mo * num_comparisons, 1.0) 
            print(f"AW vs MO: p={p_aw_mo:.4f}, p-ajustado={p_adj_aw_mo:.4f}{' *' if p_adj_aw_mo < 0.05 else ''}")

            stat_mi_mo, p_mi_mo = wilcoxon(data_mi, data_mo)
            p_adj_mi_mo = min(p_mi_mo * num_comparisons, 1.0) 
            print(f"MI vs MO: p={p_mi_mo:.4f}, p-ajustado={p_adj_mi_mo:.4f}{' *' if p_adj_mi_mo < 0.05 else ''}")
        else:
            print("\n No Post-Hoc needed", factor)
            
    except ValueError as e:
        print(f" Friedman error - {factor}: {e}")


print("\n Significant test VMIQ-2")

df_vmiq_scores = vmiq_data


num_comparisons = 3
bonferroni_p_threshold = 0.05 / num_comparisons

stat_ext_int, p_ext_int = ttest_rel(
    df_vmiq_scores['External_Visual'],
    df_vmiq_scores['Internal_Visual']
)
p_adj_ext_int = min(p_ext_int * num_comparisons, 1.0)
print(f"External vs Internal: t={stat_ext_int:.3f}, p={p_ext_int:.4f}, p-ajusted={p_adj_ext_int:.4f}{' *' if p_adj_ext_int < 0.05 else ''}")

stat_ext_kin, p_ext_kin = ttest_rel(
    df_vmiq_scores['External_Visual'],
    df_vmiq_scores['Kinesthetic']
)
p_adj_ext_kin = min(p_ext_kin * num_comparisons, 1.0)
print(f"External vs Kinesthetic: t={stat_ext_kin:.3f}, p={p_ext_kin:.4f}, p-ajusted={p_adj_ext_kin:.4f}{' *' if p_adj_ext_kin < 0.05 else ''}")

stat_int_kin, p_int_kin = ttest_rel(
    df_vmiq_scores['Internal_Visual'],
    df_vmiq_scores['Kinesthetic']
)
p_adj_int_kin = min(p_int_kin * num_comparisons, 1.0)
print(f"Internal vs Kinesthetic: t={stat_int_kin:.3f}, p={p_int_kin:.4f}, p-ajusted={p_adj_int_kin:.4f}{' *' if p_adj_int_kin < 0.05 else ''}")




 Significant test NASA

Friedman
 Value=13.2661, p-valor=0.0013
\Post-Hoc (Wilcoxon)  Mental Demand
AW vs MI: p=0.0008, p-ajustado=0.0024 *
AW vs MO: p=0.8505, p-ajustado=1.0000
MI vs MO: p=0.0017, p-ajustado=0.0050 *

Friedman
 Value=0.3721, p-valor=0.8302

 No Post-Hoc needed Physical Demand

Friedman
 Value=0.9333, p-valor=0.6271

 No Post-Hoc needed Temporal Demand

Friedman
 Value=2.5333, p-valor=0.2818

 No Post-Hoc needed Performance

Friedman
 Value=5.4727, p-valor=0.0648

 No Post-Hoc needed Effort

Friedman
 Value=0.1728, p-valor=0.9172

 No Post-Hoc needed Frustration

 Significant test VMIQ-2
External vs Internal: t=-1.706, p=0.0992, p-ajusted=0.2975
External vs Kinesthetic: t=-0.048, p=0.9619, p-ajusted=1.0000
Internal vs Kinesthetic: t=1.803, p=0.0822, p-ajusted=0.2466


C:\Users\JARS\AppData\Roaming\Python\Python311\site-packages\scipy\stats\_axis_nan_policy.py:531: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to normal approximation.



In [131]:
df = pd.read_excel('offline_classification_results.xlsx')
df_classifiers = df.loc[:,['Subject', 'Condition', 'Feature_Extraction', 'Accuracy']]
df_classifiers.rename(columns={'Subject': 'ID', 'Condition': 'TASK'}, inplace=True)
df_classifiers.Feature_Extraction = df.Feature_Extraction.map(lambda x: 'Classification_' + x)
df_classifiers


,ID,TASK,Feature_Extraction,Accuracy
0,1,AW,Classification_FBCSP,0.4125
1,2,AW,Classification_FBCSP,0.5125
2,3,AW,Classification_FBCSP,0.3750
3,4,AW,Classification_FBCSP,0.5000
4,5,AW,Classification_FBCSP,0.5375
...,...,...,...,...
256,25,MI,Classification_Combined,0.7000
257,26,MI,Classification_Combined,0.6875
258,27,MI,Classification_Combined,0.6875
259,28,MI,Classification_Combined,0.5375


In [132]:
df_class = df_classifiers.pivot_table(
	index=['ID', 'TASK'],
	columns='Feature_Extraction',
	values='Accuracy',
	aggfunc='mean'
).reset_index()

df_class.columns.name = None

df_class

,ID,TASK,Classification_Combined,Classification_ERP,Classification_FBCSP
0,1,AW,0.6000,0.5250,0.4125
1,1,MI,0.6500,0.5500,0.6625
2,1,MO,0.7125,0.4875,0.5625
3,2,AW,0.4500,0.4500,0.5125
4,2,MI,0.6875,0.5500,0.5625
...,...,...,...,...,...
82,28,MI,0.5375,0.5625,0.6250
83,28,MO,0.4875,0.4625,0.5125
84,29,AW,0.6625,0.5250,0.4125
85,29,MI,0.6875,0.4875,0.6000


In [133]:
#Wide
root_data = 'CSV_ERSP_CH/' 

freqs = np.linspace(4, 40, 80)
times = np.linspace(-1000, 3000, 200)
alpha_band = (8, 13)
beta_band = (16, 24)
time_window = (500, 2500)

time_idx_window = np.where((times >= time_window[0]) & (times <= time_window[1]))[0]
freq_idx_alpha = np.where((freqs >= alpha_band[0]) & (freqs <= alpha_band[1]))[0]
freq_idx_beta = np.where((freqs >= beta_band[0]) & (freqs <= beta_band[1]))[0]


try:
    files = os.listdir(root_data)
    files = [f for f in files if f.endswith('.csv')]
    if not files:
        print(f"No files found in {root_data}")
        exit()
except FileNotFoundError:
    print(f"Error: Directory does not exist: {root_data}")
    exit()

channels = ['C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8']
body_parts = ['Arm', 'Leg']

unique_pairs = set()
for f in files:
    try:
        parts = f.split('.')[0].split('_') 
        id_val = int(parts[-1])
        task_val = parts[1]
        unique_pairs.add((id_val, task_val))
    except (IndexError, ValueError):
        print(f"Fail: '{f}'")
        

all_data = [] 

for (id_val, task_val) in sorted(list(unique_pairs)):
    
    row = {'ID': id_val, 'TASK': task_val}
    
    for channel in channels:
        for body_part in body_parts:
            
            found_file = None
            for f in files:
                try:
                    f_parts = f.split('.')[0].split('_')
                    f_channel = f_parts[0]
                    f_task = f_parts[1]
                    f_body_part = f_parts[2]
                    f_id = int(f_parts[-1])
                    
                    if (f_id == id_val and 
                        f_task == task_val and 
                        f_channel == channel and 
                        f_body_part == body_part):
                        
                        found_file = f
                        break
                except (IndexError, ValueError):
                    continue 
            
            if found_file:
                try:
                    dff = pd.read_csv(os.path.join(root_data, found_file), header=None)
                    
                    dff.index = times
                    dff.columns = freqs
                    
                    alpha_data = dff.iloc[time_idx_window, freq_idx_alpha]
                    beta_data = dff.iloc[time_idx_window, freq_idx_beta]
                    
                    row[f'{body_part.upper()}_{channel}_Alpha_mean'] = alpha_data.mean().mean()
                    row[f'{body_part.upper()}_{channel}_Alpha_peak'] = alpha_data.mean().min()
                    row[f'{body_part.upper()}_{channel}_Beta_mean'] = beta_data.mean().mean()
                    row[f'{body_part.upper()}_{channel}_Beta_peak'] = beta_data.mean().min()
                
                except Exception as e:
                    print(f"Fail processing '{found_file}': {e}")
                    row[f'{body_part.upper()}_{channel}_Alpha_mean'] = np.nan
                    row[f'{body_part.upper()}_{channel}_Alpha_peak'] = np.nan
                    row[f'{body_part.upper()}_{channel}_Beta_mean'] = np.nan
                    row[f'{body_part.upper()}_{channel}_Beta_peak'] = np.nan

            else:
                print('File not found')
                row[f'{body_part.upper()}_{channel}_Alpha_mean'] = np.nan
                row[f'{body_part.upper()}_{channel}_Alpha_peak'] = np.nan
                row[f'{body_part.upper()}_{channel}_Beta_mean'] = np.nan
                row[f'{body_part.upper()}_{channel}_Beta_peak'] = np.nan
                
    all_data.append(row)

df_ERSP = pd.DataFrame(all_data)
df_ERSP

,ID,TASK,ARM_C3_Alpha_mean,ARM_C3_Alpha_peak,ARM_C3_Beta_mean,ARM_C3_Beta_peak,LEG_C3_Alpha_mean,LEG_C3_Alpha_peak,LEG_C3_Beta_mean,LEG_C3_Beta_peak,...,LEG_Pz_Beta_mean,LEG_Pz_Beta_peak,ARM_PO8_Alpha_mean,ARM_PO8_Alpha_peak,ARM_PO8_Beta_mean,ARM_PO8_Beta_peak,LEG_PO8_Alpha_mean,LEG_PO8_Alpha_peak,LEG_PO8_Beta_mean,LEG_PO8_Beta_peak
0,1,AW,0.399352,-0.681145,0.139322,-1.060100,0.878761,-0.392756,-0.112338,-1.363000,...,-0.603928,-1.414524,0.804728,0.058965,-0.152128,-0.863864,0.729099,-0.167563,0.276963,-0.935058
1,1,MI,1.186437,0.925100,-0.056624,-1.169782,1.694368,1.070267,-0.341806,-1.205688,...,-0.401953,-1.196974,-0.043162,-0.257882,-0.260586,-0.997847,3.296468,2.188491,-0.337788,-1.377394
2,1,MO,-1.322895,-2.128694,-0.234127,-1.535827,0.981231,0.236212,0.695517,0.333322,...,-0.074524,-0.418469,-0.205543,-1.132556,-0.336333,-1.525921,0.136974,-0.895572,0.324249,-0.086760
3,2,AW,-2.175404,-3.502153,-3.853796,-4.587610,-0.477499,-1.027564,-1.272381,-3.585947,...,-2.404018,-4.853133,-2.993437,-5.257056,-5.315836,-6.159956,0.055366,-0.463926,-2.439528,-5.395457
4,2,MI,-0.232195,-0.841505,-0.625161,-1.158329,-0.455779,-1.376475,-0.023274,-0.681563,...,-0.319956,-0.970022,0.499004,0.014685,-0.277394,-0.823998,-0.604864,-1.115088,0.059850,-0.173454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,28,MI,-3.021585,-4.151616,-1.801225,-2.772208,-1.213345,-2.050526,-0.582151,-1.355709,...,-0.748519,-1.857474,-1.623406,-1.818133,-0.406208,-1.580032,0.155174,-2.018000,-0.343008,-1.121146
83,28,MO,-0.186906,-0.814676,0.101798,-0.143073,0.969486,0.440228,0.414531,-0.180158,...,-0.010279,-0.519795,-0.807041,-1.195730,-0.089928,-0.581066,0.596925,-0.002676,-0.001968,-0.398927
84,29,AW,-0.742578,-1.096490,-0.786002,-1.176985,1.587537,1.309801,1.213828,0.860279,...,0.248826,-0.393722,0.180379,-0.566600,-0.579147,-1.212928,-0.174867,-0.526487,0.384479,0.131598
85,29,MI,0.432194,-0.097471,-0.884198,-1.746696,-0.804045,-1.816212,0.346287,0.047388,...,-1.075432,-1.284369,0.241733,-1.021178,1.277581,0.270168,1.223501,-0.224772,-0.131548,-0.745383


In [134]:
#wide
root_data = 'ERP_exports/'

channels = ['Fz', 'C3', 'Cz', 'C4', 'Pz', 'Oz']
body_parts = ['Arm', 'Leg']
feature_names = ['BP_mean', 'BP_peak', 'BP_latencypeak', 
                 'MP_mean', 'MP_peak', 'MP_latencypeak']

bp_window = (-1000, -100) 
mp_window = (-200, 500)


def extract_erp_features(df_erp, bp_window, mp_window):

    features = {}
    
    df_bp = df_erp[(df_erp['Time'] >= bp_window[0]) & (df_erp['Time'] <= bp_window[1])]
    df_mp = df_erp[(df_erp['Time'] >= mp_window[0]) & (df_erp['Time'] <= mp_window[1])]
    
    if not df_bp.empty:
        features['BP_mean'] = df_bp['ERP'].mean()
        features['BP_peak'] = df_bp['ERP'].min()
        features['BP_latencypeak'] = df_bp.loc[df_bp['ERP'].idxmin()]['Time']
    else:
        features['BP_mean'] = np.nan
        features['BP_peak'] = np.nan
        features['BP_latencypeak'] = np.nan

    if not df_mp.empty:
        features['MP_mean'] = df_mp['ERP'].mean()
        features['MP_peak'] = df_mp['ERP'].min()
        features['MP_latencypeak'] = df_mp.loc[df_mp['ERP'].idxmin()]['Time']
    else:
        features['MP_mean'] = np.nan
        features['MP_peak'] = np.nan
        features['MP_latencypeak'] = np.nan

    return features

try:
    files = os.listdir(root_data)
    files = [f for f in files if f.endswith('.csv')]
    if not files:
        print(f"Error: No .csv files in directory {root_data}")
        exit()
except FileNotFoundError:
    print(f"Error: No directory found {root_data}")
    exit()

unique_pairs = set()
for f in files:
    try:
        parts = f.split('.')[0].split('_') 
        id_val = int(parts[0])
        task_val = parts[2]
        unique_pairs.add((id_val, task_val))
    except (IndexError, ValueError):
        print(f"File {f}' with wrong format.")
        

all_data = [] # 

for (id_val, task_val) in sorted(list(unique_pairs)):
    
    row = {'ID': id_val, 'TASK': task_val}
    
    for channel in channels:
        for body_part in body_parts:
            
            expected_file = f"{id_val}_{channel}_{task_val}_{body_part.upper()}.csv"
            file_path = os.path.join(root_data, expected_file)
            
            if os.path.exists(file_path):
                try:
                    df_erp = pd.read_csv(file_path)
                    
                    features = extract_erp_features(df_erp, bp_window, mp_window)
                    
                    for key, value in features.items():
                        col_name = f"{body_part.upper()}_{channel}_{key}"
                        row[col_name] = value
                
                except Exception as e:
                    print(f"Error processing file '{expected_file}': {e}")
                    for name in feature_names:
                        col_name = f"{body_part.upper()}_{channel}_{key}"
                        row[col_name] = np.nan
            else:

                for name in feature_names:
                    col_name = f"{body_part.upper()}_{channel}_{key}"
                    row[col_name] = np.nan
    
    all_data.append(row)

df_ERP = pd.DataFrame(all_data)


df_ERP

,ID,TASK,ARM_Fz_BP_mean,ARM_Fz_BP_peak,ARM_Fz_BP_latencypeak,ARM_Fz_MP_mean,ARM_Fz_MP_peak,ARM_Fz_MP_latencypeak,LEG_Fz_BP_mean,LEG_Fz_BP_peak,...,ARM_Oz_BP_latencypeak,ARM_Oz_MP_mean,ARM_Oz_MP_peak,ARM_Oz_MP_latencypeak,LEG_Oz_BP_mean,LEG_Oz_BP_peak,LEG_Oz_BP_latencypeak,LEG_Oz_MP_mean,LEG_Oz_MP_peak,LEG_Oz_MP_latencypeak
0,1,AW,-0.486033,-5.621714,-680.0,-1.023521,-6.750661,20.0,-0.724627,-5.647821,...,-132.0,1.836565,-2.649890,-132.0,-0.517378,-4.063036,-756.0,-0.801759,-6.012942,196.0
1,1,MI,0.497026,-2.209971,-212.0,1.098760,-2.537647,500.0,0.209731,-2.642835,...,-940.0,0.054424,-4.283050,332.0,-0.455401,-3.677863,-840.0,-0.336166,-5.465744,200.0
2,1,MO,-0.429169,-2.973171,-100.0,-0.179896,-8.173830,324.0,-0.124766,-2.970369,...,-988.0,0.922380,-6.488533,132.0,-0.286355,-2.506620,-812.0,0.099030,-5.715261,136.0
3,2,AW,-0.226585,-4.549904,-896.0,-0.816872,-3.309539,288.0,-0.226140,-4.713990,...,-512.0,1.441733,-1.317041,232.0,0.210462,-2.270825,-424.0,0.824559,-1.377555,500.0
4,2,MI,-1.267074,-4.443719,-728.0,-0.136416,-6.794131,492.0,-0.893533,-5.361032,...,-532.0,-0.539195,-5.069141,284.0,-0.411275,-2.880514,-616.0,-1.037844,-5.432548,208.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,28,MI,1.567171,-1.957688,-392.0,1.381418,-2.952425,172.0,3.561097,-1.786051,...,-716.0,0.820201,-3.956689,364.0,-2.123447,-7.645370,-700.0,-2.052245,-8.698390,212.0
83,28,MO,-0.185899,-6.511583,-548.0,0.654448,-7.814500,284.0,0.147001,-4.201324,...,-112.0,2.225268,-4.478323,-68.0,-1.779366,-7.350345,-608.0,-0.456562,-4.653960,220.0
84,29,AW,0.086122,-4.362587,-764.0,-0.247231,-4.109517,40.0,1.836439,-3.534633,...,-332.0,-0.112270,-2.512877,60.0,-0.404081,-5.587075,-540.0,0.072138,-3.604185,448.0
85,29,MI,0.170995,-6.657495,-588.0,0.064782,-6.267735,40.0,0.139139,-6.134058,...,-524.0,-0.460884,-5.819525,420.0,-1.954888,-5.924464,-832.0,-1.314333,-6.737257,240.0


In [ ]:
#Long
root_data = 'CSV_ERSP_CH/'
# --- 2. Parámetros ---
freqs = np.linspace(4, 40, 80)
times = np.linspace(-1000, 3000, 200)
alpha_band = (8, 13)
beta_band = (16, 24)
time_window = (500, 2500)

time_idx_window = np.where((times >= time_window[0]) & (times <= time_window[1]))[0]
freq_idx_alpha = np.where((freqs >= alpha_band[0]) & (freqs <= alpha_band[1]))[0]
freq_idx_beta = np.where((freqs >= beta_band[0]) & (freqs <= beta_band[1]))[0]

# --- 3. Función de Extracción (Nombre corregido) ---
def extract_ersp_features(df_ersp, time_idx_window, freq_idx_alpha, freq_idx_beta):
    features = {}
    
    alpha_data = df_ersp.iloc[time_idx_window, freq_idx_alpha]
    beta_data = df_ersp.iloc[time_idx_window, freq_idx_beta]
    
    # Comprobar si los dataframes NO están vacíos
    if not alpha_data.empty and not beta_data.empty:
        features['Alpha_Mean'] = alpha_data.mean().mean()
        features['Beta_Mean'] = beta_data.mean().mean()
        features['Alpha_Peak'] = alpha_data.mean().min()
        features['Beta_Peak'] = beta_data.mean().min()
    else:
        features['Alpha_Mean'] = np.nan
        features['Beta_Mean'] = np.nan
        features['Alpha_Peak'] = np.nan
        features['Beta_Peak'] = np.nan
    return features

# --- 4. Carga de Archivos ---
try:
    files = os.listdir(root_data)
    files = [f for f in files if f.endswith('.csv')]
    if not files:
        print(f"Advertencia: No se encontraron archivos .csv en el directorio {root_data}")
        exit()
except FileNotFoundError:
    print(f"Error: El directorio no existe: {root_data}")
    exit()

channels = ['C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8']
body_parts = ['Arm', 'Leg']

# --- 5. Encontrar Pares Únicos ---
unique_pairs = set()
for f in files:
    try:
        parts = f.split('.')[0].split('_') 
        id_val = int(parts[-1])
        task_val = parts[1]
        unique_pairs.add((id_val, task_val))
    except (IndexError, ValueError):
        print(f"Advertencia: El archivo '{f}' no sigue el formato [Canal]_[Task]_[Parte]_[...]. Se omitirá.")
        
# --- 6. Bucle de Extracción (Lógica CORREGIDA) ---
all_data = [] 

for (id_val, task_val) in sorted(list(unique_pairs)):
    for channel in channels:
        for body_part in body_parts:
            
            # Construir nombre de archivo esperado
            found_file = None
            for f in files:
                try:
                    f_parts = f.split('.')[0].split('_')
                    f_channel = f_parts[0]
                    f_task = f_parts[1]
                    f_body_part = f_parts[2]
                    f_id = int(f_parts[-1])
                    
                    if (f_id == id_val and 
                        f_task == task_val and 
                        f_channel == channel and 
                        f_body_part == body_part):
                        
                        found_file = f
                        break
                except (IndexError, ValueError):
                    continue 
            
            if found_file:
                try:
                    dff = pd.read_csv(os.path.join(root_data, found_file), header=None)
                    
                    # --- Validación de Dimensiones ---
                    if dff.shape != (200, 80):
                         print(f"Error: Dimensiones incorrectas en '{found_file}'. Se esperan (200, 80) pero se obtuvieron {dff.shape}. Omitiendo.")
                         continue # Omitir este archivo
                         
                    dff.index = times
                    dff.columns = freqs
                    features = extract_ersp_features(dff, time_idx_window, freq_idx_alpha, freq_idx_beta)
                    
                    # --- LÓGICA CORREGIDA ---
                    # Iterar sobre las características extraídas y AÑADIR UNA NUEVA FILA POR CADA UNA
                    for key, value in features.items():
                        row = {
                            'ID': id_val,
                            'TASK': task_val,
                            'CH': channel,
                            'Limb': body_part.upper(),
                            'Feature': key,  # E.g., 'Alpha_Mean', 'Beta_Peak'
                            'value': value
                        }
                        all_data.append(row) # <-- Append SE MUEVE AQUÍ

                except Exception as e:
                    print(f"Error procesando el archivo '{found_file}': {e}")
                    # No añadir nada si el archivo falla
            
            else:
                # Opcional: Si quieres añadir filas NaN para datos faltantes
                # print(f"Info: No se encontró archivo para ID={id_val}, TASK={task_val}, CH={channel}, Limb={body_part}")
                for key in ['Alpha_Mean', 'Beta_Mean', 'Alpha_Peak', 'Beta_Peak']:
                    row = {
                        'ID': id_val,
                        'TASK': task_val,
                        'CH': channel,
                        'Limb': body_part.upper(),
                        'Feature': key,
                        'value': np.nan # Rellenar con NaN
                    }
                    all_data.append(row)

# --- 7. Crear DataFrame Final ---
df_ERSP = pd.DataFrame(all_data)
df_ERSP

In [ ]:
#long
root_data = 'ERP_exports/'
channels = ['Fz', 'C3', 'Cz', 'C4', 'Pz', 'Oz']
body_parts = ['Arm', 'Leg']
feature_names = ['BP_Mean', 'BP_Peak', 'BP_LatencyPeak', 
                 'MP_Mean', 'MP_Peak', 'MP_LatencyPeak']
bp_window = (-1000, -100) 
mp_window = (-200, 500)

# --- 3. Función de Extracción ---
def extract_erp_features(df_erp, bp_window, mp_window):
    features = {}
    df_bp = df_erp[(df_erp['Time'] >= bp_window[0]) & (df_erp['Time'] <= bp_window[1])]
    df_mp = df_erp[(df_erp['Time'] >= mp_window[0]) & (df_erp['Time'] <= mp_window[1])]
    
    if not df_bp.empty:
        features['BP_Mean'] = df_bp['ERP'].mean()
        features['BP_Peak'] = df_bp['ERP'].min()
        features['BP_LatencyPeak'] = df_bp.loc[df_bp['ERP'].idxmin()]['Time']
    else:
        features['BP_Mean'] = np.nan
        features['BP_Peak'] = np.nan
        features['BP_LatencyPeak'] = np.nan
        
    if not df_mp.empty:
        features['MP_Mean'] = df_mp['ERP'].mean()
        features['MP_Peak'] = df_mp['ERP'].min()
        features['MP_LatencyPeak'] = df_mp.loc[df_mp['ERP'].idxmin()]['Time']
    else:
        features['MP_Mean'] = np.nan
        features['MP_Peak'] = np.nan
        features['MP_LatencyPeak'] = np.nan
    return features

# --- 4. Carga de Archivos ---
try:
    files = os.listdir(root_data)
    files = [f for f in files if f.endswith('.csv')]
    if not files:
        print(f"Advertencia: No se encontraron archivos .csv en el directorio {root_data}")
        exit()
except FileNotFoundError:
    print(f"Error: El directorio no existe: {root_data}")
    exit()

# --- 5. Encontrar Pares Únicos ---
unique_pairs = set()
for f in files:
    try:
        parts = f.split('.')[0].split('_')
        id_val = int(parts[0])
        task_val = parts[2]
        unique_pairs.add((id_val, task_val))
    except (IndexError, ValueError):
        print(f"Advertencia: El archivo '{f}' no sigue el formato [ID]_[Canal]_[Task]_[Parte]. Se omitirá.")
        
# --- 6. Bucle de Extracción (Lógica CORREGIDA) ---
all_data = [] 

for (id_val, task_val) in sorted(list(unique_pairs)):
    for channel in channels:
        for body_part in body_parts:
            
            # Corregir el nombre del archivo esperado para que coincida con TUS mayúsculas (ARM/LEG)
            expected_file = f"{id_val}_{channel}_{task_val}_{body_part.upper()}.csv"
            file_path = os.path.join(root_data, expected_file)
            
            if os.path.exists(file_path):
                try:
                    df_erp = pd.read_csv(file_path)
                    features = extract_erp_features(df_erp, bp_window, mp_window)
                    
                    # --- LÓGICA CORREGIDA ---
                    # Iterar sobre las características extraídas y AÑADIR UNA NUEVA FILA POR CADA UNA
                    for key, value in features.items():
                        row = {
                            'ID': id_val,
                            'TASK': task_val,
                            'CH': channel,
                            'Limb': body_part.upper(),
                            'Feature': key, # E.g., 'BP_Mean', 'MP_Peak'
                            'value': value
                        }
                        all_data.append(row) # <-- Append SE MUEVE AQUÍ

                except Exception as e:
                    print(f"Error procesando el archivo '{expected_file}': {e}")
            
            else:
                # Opcional: Si quieres añadir filas NaN para datos faltantes
                # print(f"Info: No se encontró archivo: {expected_file}")
                for key in feature_names:
                    row = {
                        'ID': id_val,
                        'TASK': task_val,
                        'CH': channel,
                        'Limb': body_part.upper(),
                        'Feature': key,
                        'value': np.nan # Rellenar con NaN
                    }
                    all_data.append(row)

# --- 7. Crear DataFrame Final ---
df_ERP = pd.DataFrame(all_data)
df_ERP

In [136]:
df_NASA

,ID,TASK,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration
0,1,AW,140,500,160,0,300,70
1,2,AW,10,50,30,300,30,0
2,3,AW,200,0,240,240,120,40
3,4,AW,250,20,120,320,160,0
4,5,AW,400,180,50,180,320,0
...,...,...,...,...,...,...,...,...
82,25,MO,150,0,120,140,70,400
83,26,MO,350,160,160,60,200,70
84,27,MO,120,0,40,90,60,60
85,28,MO,240,0,60,320,160,10


In [146]:
df_VIQ.columns = df_VIQ.columns[0:1].to_list() + df_VIQ.columns[1:].map(lambda x: 'VIQ_'+x).to_list()
df_NASA.columns = df_NASA.columns[0:2].to_list() + df_NASA.columns[2:].map(lambda x: 'NASA_'+x).to_list()


In [149]:
df_VIQ_NASA_ACC_ERSP_ERP= df_VIQ.merge(df_NASA, on='ID').merge(df_class.merge(df_ERSP.merge(df_ERP, on=['ID','TASK']), on=['ID','TASK']), on=['ID', 'TASK'])
df_VIQ_NASA_ACC_ERSP_ERP

,ID,VIQ_External_Visual,VIQ_Internal_Visual,VIQ_Kinesthetic,TASK,NASA_Mental Demand,NASA_Physical Demand,NASA_Temporal Demand,NASA_Performance,NASA_Effort,...,ARM_Oz_BP_latencypeak,ARM_Oz_MP_mean,ARM_Oz_MP_peak,ARM_Oz_MP_latencypeak,LEG_Oz_BP_mean,LEG_Oz_BP_peak,LEG_Oz_BP_latencypeak,LEG_Oz_MP_mean,LEG_Oz_MP_peak,LEG_Oz_MP_latencypeak
0,1,4.833333,3.833333,5.000000,AW,140,500,160,0,300,...,-132.0,1.836565,-2.649890,-132.0,-0.517378,-4.063036,-756.0,-0.801759,-6.012942,196.0
1,1,4.833333,3.833333,5.000000,MI,240,180,200,90,270,...,-940.0,0.054424,-4.283050,332.0,-0.455401,-3.677863,-840.0,-0.336166,-5.465744,200.0
2,1,4.833333,3.833333,5.000000,MO,280,30,150,140,210,...,-988.0,0.922380,-6.488533,132.0,-0.286355,-2.506620,-812.0,0.099030,-5.715261,136.0
3,2,3.000000,4.000000,3.916667,AW,10,50,30,300,30,...,-512.0,1.441733,-1.317041,232.0,0.210462,-2.270825,-424.0,0.824559,-1.377555,500.0
4,2,3.000000,4.000000,3.916667,MI,20,80,200,180,60,...,-532.0,-0.539195,-5.069141,284.0,-0.411275,-2.880514,-616.0,-1.037844,-5.432548,208.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,28,4.833333,4.916667,4.583333,MI,120,0,150,400,60,...,-716.0,0.820201,-3.956689,364.0,-2.123447,-7.645370,-700.0,-2.052245,-8.698390,212.0
83,28,4.833333,4.916667,4.583333,MO,240,0,60,320,160,...,-112.0,2.225268,-4.478323,-68.0,-1.779366,-7.350345,-608.0,-0.456562,-4.653960,220.0
84,29,3.916667,3.666667,3.250000,AW,270,0,150,280,60,...,-332.0,-0.112270,-2.512877,60.0,-0.404081,-5.587075,-540.0,0.072138,-3.604185,448.0
85,29,3.916667,3.666667,3.250000,MI,360,0,60,360,210,...,-524.0,-0.460884,-5.819525,420.0,-1.954888,-5.924464,-832.0,-1.314333,-6.737257,240.0


In [152]:
subjetive_variables = df_VIQ.drop(columns='ID', errors='ignore').columns.to_list() + df_NASA.drop(columns=['ID', 'TASK'], errors='ignore').columns.to_list()
classification_variables =  df_class.drop(columns=['ID', 'TASK'], errors='ignore').columns.to_list()
freq_variables = df_ERSP.drop(columns=['ID', 'TASK'], errors='ignore').columns.to_list()
time_variables = df_ERP.drop(columns=['ID', 'TASK'], errors='ignore').columns.to_list()

objective_variables = classification_variables+freq_variables+time_variables

In [153]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau


df_master = df_VIQ_NASA_ACC_ERSP_ERP.copy()

corr_matrix={}
for task in ['AW', 'MI', 'MO']:
    corr_matrix[task]={}
# 1. Filtrar tu DataFrame por la condición 'AW'
    df = df_master[df_master['TASK'] == task]
    df = df.drop(columns=['ID', 'TASK'], errors='ignore')

    corr_matrix[task]['tau'] = pd.DataFrame(index=subjetive_variables,
                                            columns=objective_variables, 
                                            dtype=float)
    corr_matrix[task]['pvalue'] = pd.DataFrame(index=subjetive_variables,
                                            columns=objective_variables, 
                                            dtype=float)
    for row in subjetive_variables:
        for col in objective_variables:
            tau, p = kendalltau(df[row], df[col])
            corr_matrix[task]['tau'].loc[row, col] = tau
            corr_matrix[task]['pvalue'].loc[row, col] = p



In [ ]:
significant_df={}
for task in tasks:
    significant_indices = np.where(corr_matrix[task]['pvalue'].values < 0.05)
    significant_df[task] =pd.DataFrame({
    'Subjective_Variable': corr_matrix[task]['pvalue'].index[significant_indices[0]],
    'Objective_Variable': corr_matrix[task]['pvalue'].columns[significant_indices[1]],
    'tau': corr_matrix[task]['tau'].values[significant_indices],
    'pvalue': corr_matrix[task]['pvalue'].values[significant_indices]
}).sort_values(by='tau', ascending=True).reset_index().drop(columns='index')


In [ ]:

colors = {
    'AW': '#ef8a62',
    'MO': '#67a9cf',
    'MI': '#999999',
    'AW_2': '#613828'
}
task_colors = {
    'AW': colors['AW'],
    'MI': colors['MI'],
    'MO': colors['MO']
}
tasks = ['AW', 'MI', 'MO']

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[
        "Subjective Variables", "Objective Variables", "Channel",
        "", "","",
        "", "", "",

    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

for i, task in enumerate(tasks, start=1):
    df = significant_df[task]
    # Subjective variable counts
    subj_counts = df['Subjective_Variable'].value_counts().reset_index()
    subj_counts_sorted = subj_counts.sort_values('Subjective_Variable')
    fig.add_trace(
        go.Bar(
            y=subj_counts_sorted['index'],
            x=subj_counts_sorted['Subjective_Variable'],
            text=subj_counts_sorted['Subjective_Variable'],
            marker_color=task_colors[task],
            name=f'{task}',
            orientation='h'
        ),
        row=i, col=1
    )
    # Objective variable counts
    obj_series = df['Objective_Variable']
    counts = {
        'ARM': obj_series.str.contains('ARM').sum(),
        'LEG': obj_series.str.contains('LEG').sum(),
        'Alpha': obj_series.str.contains('Alpha').sum(),
        'Beta': obj_series.str.contains('Beta').sum(),
        'BP': obj_series.str.contains('BP').sum(),
        'MP': obj_series.str.contains('MP').sum(),
        'Mean': obj_series.str.contains('mean', case=False).sum(),
        'Peak': obj_series.str.contains('_peak', case=False).sum(),
        'Latency': obj_series.str.contains('latency', case=False).sum()
    }
    counts_sorted = dict(sorted(counts.items(), key=lambda x: x[1]))
    fig.add_trace(
        go.Bar(
            y=list(counts_sorted.keys()),
            x=list(counts_sorted.values()),
            text=list(counts_sorted.values()),
            marker_color=task_colors[task],
            name=f'{task} ob',
            orientation='h'
        ),
        row=i, col=2
    )
    # Channel counts
    channels = ['C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8', 'Fz', 'Oz']
    ch_count = {ch: obj_series.str.contains(ch).sum() for ch in channels}
    ch_count_sorted = dict(sorted(ch_count.items(), key=lambda x: x[1]))
    fig.add_trace(
        go.Bar(
            y=list(ch_count_sorted.keys()),
            x=list(ch_count_sorted.values()),
            text=list(ch_count_sorted.values()),
            marker_color=task_colors[task],
            name=f'{task} ch',
            orientation='h'
        ),
        row=i, col=3
    )

fig.update_layout(
    height=1200, width=1600,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="center",
        x=0.5,
        itemsizing='constant'
    ),
    template="simple_white"
)
# Only keep legend entries for MO, MI, AW (one per task)
for trace in fig.data:
    if trace.name not in ['MO', 'MI', 'AW']:
        trace.showlegend = False
fig.update_xaxes(title='count', row=3, col=3)
fig.show()


: 

In [158]:
with pd.ExcelWriter('significant_correlations.xlsx') as writer:
    for task in ['MO', 'MI', 'AW']:
        significant_df[task].to_excel(writer, sheet_name=task, index=False)